In [1]:
import sys,os
sys.path.append(os.getcwd())
from src.utils.data_utils import round_to_closest_indices, make_padding
import logging
import numpy as np
import pandas as pd
import torch
from omegaconf import DictConfig, OmegaConf
import yaml
import hashlib
from numpy.lib.stride_tricks import sliding_window_view
from datetime import datetime
import time
import polars
import gc
from functools import partial
import glob 
from pytorch_lightning.utilities import rank_zero_only
import pickle

/home/a.galliamov/miniconda3/envs/wind/lib/python3.10/site-packages/lightning_fabric/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


In [5]:
time_coords = np.load(os.path.join('data/cmip5_world/', 'time.npy')).astype('datetime64[D]')
lat_coords = np.load(os.path.join('data/cmip5_world/', 'lat.npy'))
lon_coords = np.load(os.path.join('data/cmip5_world/', 'lon.npy'))

def time_to_data_grid(target_df):
    """перевести реальные временные метки станционных наблюдений в индексы ближайших узлов временной сетки модели"""
    start_time = time.process_time()   
    dates = target_df["time"].to_numpy()
    y = target_df["y"].to_numpy()
    values = round_to_closest_indices(dates, time_coords) # time into inds
    target_df = target_df.with_columns(
                        polars.Series(name="time", values=values),
                        polars.Series(name="y", values=y),
                        )
    print(f"Time align took {time.process_time() - start_time} seconds")
    return target_df


def stations_to_data_grid(stations_df: polars.DataFrame) -> polars.DataFrame:
    """ maps stations to the data grid pixels """
    start_time = time.process_time()   
    lat = stations_df["lat"].to_numpy() 
    lon = stations_df["lon"].to_numpy()
    lat_vector = round_to_closest_indices(lat, lat_coords)
    lon_vector = round_to_closest_indices(lon, lon_coords)
    stations_df = stations_df.with_columns(
                        [
                            polars.Series(name="lat", values=lat_vector),
                            polars.Series(name="lon", values=lon_vector)
                        ])
    print(f"Closest pixel search took {time.process_time() - start_time} seconds")
    return stations_df

In [6]:
def prepare_target_df():
    target_df = polars.read_parquet(os.path.join('data/cmip5_world/', 'target.parquet')) # data/cmip5_world/target.parquet - станционные данные
    print(f"Records before preparation {len(target_df)}")
    start_date = pd.to_datetime('2000-01-01')
    end_date = pd.to_datetime('2018-12-31')
    print(f"Target time bounds before filter {target_df['time'].min()}, {target_df['time'].max()}")
    target_df = target_df.filter((polars.col('time') >= start_date) & (polars.col('time') < end_date)) # Может <= ??????
    print(f"Target time bounds after filter {target_df['time'].min()}, {target_df['time'].max()}")
    print(f"Data time bounds {time_coords.min()}, {time_coords.max()}") # self.time_coords = np.load(os.path.join(self.cfg.train.data_dir, 'time.npy')).astype('datetime64[D]')
    print(f"Stations before aggregation: {target_df.n_unique(subset=['lat', 'lon'])}")
    target_df = time_to_data_grid(target_df)
    target_df = stations_to_data_grid(target_df)

    target_df_1 = (target_df
                .lazy()        
                .sort("time")
                .group_by(["lat", "lon", "time"])
                .agg(
                    [
                        polars.col('y').quantile(0.65).alias("y"), # ?
                    ])
                .collect())
    
    target_df_2 = (target_df_1
                .lazy()        
                .sort("time")
                .group_by(["lat", "lon"])
                .agg(
                    [
                        polars.col("time"),
                        polars.col('y'),
                    ])
                .collect())
    target_df_2 = target_df_2.drop_nulls()
    print(f"Stations after aggregation: {len(target_df_2)}")

In [4]:
prepare_target_df()

In [7]:

target_df = polars.read_parquet(os.path.join('data/cmip5_world/', 'target.parquet')) # data/cmip5_world/target.parquet - станционные данные
print(f"Records before preparation {len(target_df)}")
start_date = pd.to_datetime('2000-01-01')
end_date = pd.to_datetime('2018-12-31')
print(f"Target time bounds before filter {target_df['time'].min()}, {target_df['time'].max()}")
target_df = target_df.filter((polars.col('time') >= start_date) & (polars.col('time') < end_date)) # Может <= ??????
print(f"Target time bounds after filter {target_df['time'].min()}, {target_df['time'].max()}")
print(f"Data time bounds {time_coords.min()}, {time_coords.max()}") # self.time_coords = np.load(os.path.join(self.cfg.train.data_dir, 'time.npy')).astype('datetime64[D]')
print(f"Stations before aggregation: {target_df.n_unique(subset=['lat', 'lon'])}")
target_df = time_to_data_grid(target_df)
target_df = stations_to_data_grid(target_df)

target_df_1 = (target_df
            .lazy()        
            .sort("time")
            .group_by(["lat", "lon", "time"])
            .agg(
                [
                    polars.col('y').quantile(0.65).alias("y"), # ?
                ])
            .collect())

target_df_2 = (target_df_1
            .lazy()        
            .sort("time")
            .group_by(["lat", "lon"])
            .agg(
                [
                    polars.col("time"),
                    polars.col('y'),
                ])
            .collect())
target_df_2 = target_df_2.drop_nulls()
print(f"Stations after aggregation: {len(target_df_2)}")

Records before preparation 17101686
Target time bounds before filter 2000-01-02, 2018-12-31
Target time bounds after filter 2000-01-02, 2018-12-30
Data time bounds 2000-01-01, 2018-12-30
Stations before aggregation: 3323
Time align took 1.1340669270000063 seconds
Closest pixel search took 1.6020637230000006 seconds
Stations after aggregation: 3001


In [12]:

target_df = polars.read_parquet(os.path.join('data/cmip5_world/', 'target.parquet')) # data/cmip5_world/target.parquet - станционные данные
print(f"Records before preparation {len(target_df)}")
start_date = pd.to_datetime('2000-01-01')
end_date = pd.to_datetime('2018-12-31')
print(f"Target time bounds before filter {target_df['time'].min()}, {target_df['time'].max()}")
target_df = target_df.filter((polars.col('time') >= start_date) & (polars.col('time') < end_date)) # Может <= ??????
print(f"Target time bounds after filter {target_df['time'].min()}, {target_df['time'].max()}")
print(f"Data time bounds {time_coords.min()}, {time_coords.max()}") # self.time_coords = np.load(os.path.join(self.cfg.train.data_dir, 'time.npy')).astype('datetime64[D]')
print(f"Stations before aggregation: {target_df.n_unique(subset=['lat', 'lon'])}")
target_df = time_to_data_grid(target_df)
target_df = stations_to_data_grid(target_df)

target_df_1 = (target_df
            .lazy()        
            .sort("time")
            .group_by(["lat", "lon", "time"])
            .agg(
                [
                    polars.col('y').quantile(0.99).alias("y"), # ?
                ])
            .collect())

target_df_2 = (target_df_1
            .lazy()        
            .sort("time")
            .group_by(["lat", "lon"])
            .agg(
                [
                    polars.col("time"),
                    polars.col('y'),
                ])
            .collect())
target_df_2 = target_df_2.drop_nulls()
print(f"Stations after aggregation: {len(target_df_2)}")

Records before preparation 17101686
Target time bounds before filter 2000-01-02, 2018-12-31
Target time bounds after filter 2000-01-02, 2018-12-30
Data time bounds 2000-01-01, 2018-12-30
Stations before aggregation: 3323
Time align took 1.1085364340000297 seconds
Closest pixel search took 1.7132921090000082 seconds
Stations after aggregation: 3001


In [20]:
df65 = (target_df
  .group_by(["lat","lon","time"])
  .agg(polars.col("y").quantile(0.65).alias("y65"))
)

df99 = (target_df
  .group_by(["lat","lon","time"])
  .agg(polars.col("y").quantile(0.90).alias("y90"))
)

In [21]:
diff = (
    df65.join(df99, on=["lat","lon","time"], how="inner")
       .filter(polars.col("y65") != polars.col("y90"))
       .with_columns((polars.col("y90") - polars.col("y65")).alias("delta"))
       .sort(polars.col("delta").abs(), descending=True)
)
diff.head(50)

lat,lon,time,y65,y90,delta
i64,i64,i64,f32,f32,f32
91,10,2188,12.0,51.0,39.0
87,0,1605,6.0,41.0,35.0
82,13,4342,2.0,36.0,34.0
97,7,5804,13.0,47.0,34.0
82,13,4410,3.0,36.0,33.0
…,…,…,…,…,…
82,13,4408,4.0,32.0,28.0
81,4,3332,18.0,46.0,28.0
83,10,6506,7.0,34.0,27.0


In [13]:
target_df_2

lat,lon,time,y
i64,i64,list[i64],list[f32]
71,46,"[133, 134, … 6754]","[5.0, 5.0, … 10.0]"
28,129,"[1, 2, … 6933]","[2.0, 3.0, … 11.0]"
92,73,"[521, 522, … 6933]","[6.0, 4.0, … 3.0]"
68,97,"[1, 3, … 6933]","[1.0, 1.0, … 1.0]"
96,24,"[4567, 4915, … 6701]","[7.0, 5.0, … 7.0]"
…,…,…,…
58,30,"[280, 281, … 3358]","[7.0, 3.0, … 9.0]"
82,74,"[4517, 4527, … 6933]","[6.0, 3.0, … 2.0]"
91,27,"[521, 522, … 6932]","[3.0, 3.0, … 2.0]"


In [10]:
dup_stats = (
    target_df
    .group_by(["lat","lon","time"])
    .len()
    .select([
        polars.col("len").max().alias("max_group_size"),
        (polars.col("len") > 1).sum().alias("n_groups_gt1"),
        polars.len().alias("n_groups_total"),
    ])
)
print(dup_stats)

shape: (1, 3)
┌────────────────┬──────────────┬────────────────┐
│ max_group_size ┆ n_groups_gt1 ┆ n_groups_total │
│ ---            ┆ ---          ┆ ---            │
│ u32            ┆ u32          ┆ u32            │
╞════════════════╪══════════════╪════════════════╡
│ 36             ┆ 2934258      ┆ 11562882       │
└────────────────┴──────────────┴────────────────┘


In [11]:
agg = (
    target_df
    .group_by(["lat","lon","time"])
    .agg([
        polars.col("y").count().alias("n"),
        polars.col("y").min().alias("y_min"),
        polars.col("y").max().alias("y_max"),
        polars.col("y").mean().alias("y_mean"),
        polars.col("y").quantile(0.5).alias("y_q50"),
        polars.col("y").quantile(0.65).alias("y_q65"),
        polars.col("y").quantile(0.99).alias("y_q99"),
    ])
)

# Смотрим только реальные дубликаты
agg_dup = agg.filter(polars.col("n") > 1).with_columns([
    (polars.col("y_max") - polars.col("y_q65")).alias("max_minus_q65"),
    (polars.col("y_max") - polars.col("y_q99")).alias("max_minus_q99"),
    (polars.col("y_q99") - polars.col("y_q65")).alias("q99_minus_q65"),
    (polars.col("y_max") - polars.col("y_q50")).alias("max_minus_q50"),
])

print(
    agg_dup.select([
        polars.col("n").mean().alias("avg_n"),
        polars.col("max_minus_q65").quantile(0.5).alias("median(max-q65)"),
        polars.col("max_minus_q65").quantile(0.9).alias("p90(max-q65)"),
        polars.col("max_minus_q99").quantile(0.5).alias("median(max-q99)"),
        polars.col("max_minus_q50").quantile(0.5).alias("median(max-q50)"),
    ])
)

shape: (1, 5)
┌─────────┬─────────────────┬──────────────┬─────────────────┬─────────────────┐
│ avg_n   ┆ median(max-q65) ┆ p90(max-q65) ┆ median(max-q99) ┆ median(max-q50) │
│ ---     ┆ ---             ┆ ---          ┆ ---             ┆ ---             │
│ f64     ┆ f32             ┆ f32          ┆ f32             ┆ f32             │
╞═════════╪═════════════════╪══════════════╪═════════════════╪═════════════════╡
│ 2.88658 ┆ 0.0             ┆ 3.0          ┆ 0.0             ┆ 0.0             │
└─────────┴─────────────────┴──────────────┴─────────────────┴─────────────────┘
